# Structured Query API Reference

Developer-facing statements defined in `libs/core/langchain_core/structured_query.py`.

# `Visitor: ABC`

Abstract visitor interface for translating structured-query intermediate-representation objects.

Concrete subclasses must implement `visit_operation`, `visit_comparison`, and `visit_structured_query`. The abstract methods do not explicitly raise `NotImplementedError`.

## Fields

```python
allowed_comparators: Sequence[Comparator] | None = None # Comparators allowed by the visitor
allowed_operators: Sequence[Operator] | None = None # Operators allowed by the visitor
```

## Required subclass hooks

### `visit_operation`

Translates an `Operation`.

```python
@abstractmethod
visit_operation(
    self,
    operation: Operation, # Operation to translate
) -> Any # Translated representation
```

### `visit_comparison`

Translates a `Comparison`.

```python
@abstractmethod
visit_comparison(
    self,
    comparison: Comparison, # Comparison to translate
) -> Any # Translated representation
```

### `visit_structured_query`

Translates a `StructuredQuery`.

```python
@abstractmethod
visit_structured_query(
    self,
    structured_query: StructuredQuery, # Structured query to translate
) -> Any # Translated representation
```

---

# `Expr: BaseModel`

Base class for structured-query expressions.

## Methods

### `accept`

Dispatches the expression to the matching visitor method based on the expression class name.

```python
accept(
    self,
    visitor: Visitor, # Visitor that translates the expression
) -> Any # Result returned by the matching visitor method
```

`Comparison`, `Operation`, and `StructuredQuery` dispatch to `visit_comparison`, `visit_operation`, and `visit_structured_query`, respectively.

---

# `Operator: str, Enum`

Logical operators supported by structured-query operations.

## Members

```python
AND = "and"
OR = "or"
NOT = "not"
```

---

# `Comparator: str, Enum`

Comparison operators supported by structured-query comparisons.

## Members

```python
EQ = "eq"
NE = "ne"
GT = "gt"
GTE = "gte"
LT = "lt"
LTE = "lte"
CONTAIN = "contain"
LIKE = "like"
IN = "in"
NIN = "nin"
```

# `FilterDirective: Expr, ABC`

Base class for filtering expressions.

It inherits `ABC` but declares no abstract methods of its own.


# `Comparison: FilterDirective`

Represents a comparison between an attribute and a value.

## Fields

```python
comparator: Comparator # Comparator to apply
attribute: str # Attribute to compare
value: Any # Value to compare against
```

## Constructor

```python
Comparison(
    comparator: Comparator, # Comparator to apply
    attribute: str, # Attribute to compare
    value: Any, # Value to compare against
    **kwargs: Any, # Additional values forwarded to BaseModel
) -> None
```

# `Operation: FilterDirective`

Represents a logical operation over filtering directives.

## Fields

```python
operator: Operator # Logical operator to apply
arguments: list[FilterDirective] # Filtering directives supplied to the operator
```

## Constructor

```python
Operation(
    operator: Operator, # Logical operator to apply
    arguments: list[FilterDirective], # Filtering directives supplied to the operator
    **kwargs: Any, # Additional values forwarded to BaseModel
) -> None
```

In [ ]:
from typing import Any # Import the general-purpose type

from langchain_core.structured_query import Comparator # Import comparison operators
from langchain_core.structured_query import Comparison # Import attribute comparison expressions
from langchain_core.structured_query import Operation # Import logical filter operations
from langchain_core.structured_query import Operator # Import logical operators
from langchain_core.structured_query import StructuredQuery # Import the complete structured-query model
from langchain_core.structured_query import Visitor # Import the abstract visitor interface


class MongoFilterVisitor(Visitor): # Define a visitor that converts expressions into MongoDB filters
    allowed_comparators = [ # Declare the comparison operators supported by this visitor
        Comparator.EQ, # Support equality comparisons
        Comparator.NE, # Support inequality comparisons
        Comparator.GT, # Support greater-than comparisons
        Comparator.GTE, # Support greater-than-or-equal comparisons
        Comparator.LT, # Support less-than comparisons
        Comparator.LTE, # Support less-than-or-equal comparisons
        Comparator.IN, # Support membership comparisons
        Comparator.NIN, # Support non-membership comparisons
    ] # Finish the supported comparator list

    allowed_operators = [ # Declare the logical operators supported by this visitor
        Operator.AND, # Support logical AND
        Operator.OR, # Support logical OR
        Operator.NOT, # Support logical NOT
    ] # Finish the supported operator list

    def visit_comparison(self, comparison: Comparison) -> dict[str, Any]: # Translate one comparison into a MongoDB filter
        comparator_map = { # Map LangChain comparators to MongoDB operators
            Comparator.EQ: "$eq", # Map equality to MongoDB equality
            Comparator.NE: "$ne", # Map inequality to MongoDB inequality
            Comparator.GT: "$gt", # Map greater-than to MongoDB greater-than
            Comparator.GTE: "$gte", # Map greater-than-or-equal to MongoDB greater-than-or-equal
            Comparator.LT: "$lt", # Map less-than to MongoDB less-than
            Comparator.LTE: "$lte", # Map less-than-or-equal to MongoDB less-than-or-equal
            Comparator.IN: "$in", # Map membership to MongoDB membership
            Comparator.NIN: "$nin", # Map non-membership to MongoDB non-membership
        } # Finish the comparator mapping

        if comparison.comparator not in self.allowed_comparators: # Check whether the comparator is supported
            raise ValueError(f"Unsupported comparator: {comparison.comparator}") # Reject unsupported comparisons

        mongo_operator = comparator_map[comparison.comparator] # Get the matching MongoDB operator

        return { # Return the translated comparison
            comparison.attribute: {mongo_operator: comparison.value} # Apply the operator to the attribute and value
        } # Finish the comparison filter

    def visit_operation(self, operation: Operation) -> dict[str, Any]: # Translate a logical operation into a MongoDB filter
        if operation.operator not in self.allowed_operators: # Check whether the logical operator is supported
            raise ValueError(f"Unsupported operator: {operation.operator}") # Reject unsupported logical operations

        translated_arguments = [ # Translate every nested filter expression
            argument.accept(self) for argument in operation.arguments # Dispatch each expression back to this visitor
        ] # Finish translating the nested arguments

        if operation.operator == Operator.AND: # Handle a logical AND operation
            return {"$and": translated_arguments} # Require all nested conditions to match

        if operation.operator == Operator.OR: # Handle a logical OR operation
            return {"$or": translated_arguments} # Require at least one nested condition to match

        if operation.operator == Operator.NOT: # Handle a logical NOT operation
            if len(translated_arguments) != 1: # Ensure NOT receives exactly one nested condition
                raise ValueError("NOT requires exactly one argument") # Reject invalid NOT expressions

            return {"$nor": translated_arguments} # Negate the supplied MongoDB condition

        raise ValueError(f"Unsupported operator: {operation.operator}") # Protect against unexpected operators

    def visit_structured_query(self, structured_query: StructuredQuery) -> dict[str, Any]: # Translate the complete structured query
        mongo_query: dict[str, Any] = { # Create the final translated result
            "text_query": structured_query.query, # Preserve the semantic search text
            "filter": {}, # Provide an empty filter by default
            "limit": structured_query.limit, # Preserve the requested result limit
        } # Finish the result dictionary

        if structured_query.filter is not None: # Check whether the query contains filtering instructions
            mongo_query["filter"] = structured_query.filter.accept(self) # Translate the filter through visitor dispatch

        return mongo_query # Return the translated structured query


price_filter = Comparison( # Create a price comparison expression
    Comparator.GTE, # Select greater-than-or-equal comparison
    "price", # Specify the product field
    50000, # Specify the minimum product price
) # Finish creating the price comparison

brand_filter = Comparison( # Create a brand comparison expression
    Comparator.IN, # Select membership comparison
    "brand", # Specify the brand field
    ["Dell", "Lenovo"], # Specify the accepted brands
) # Finish creating the brand comparison

stock_filter = Comparison( # Create an inventory comparison expression
    Comparator.GT, # Select greater-than comparison
    "stock", # Specify the inventory field
    0, # Require at least one available item
) # Finish creating the inventory comparison

combined_filter = Operation( # Combine the individual conditions
    Operator.AND, # Require every condition to match
    [price_filter, brand_filter, stock_filter], # Supply the nested filter expressions
) # Finish creating the logical operation

structured_query = StructuredQuery( # Create the complete structured query
    "laptop for software development", # Provide the semantic search text
    combined_filter, # Attach the structured metadata filter
    limit=5, # Restrict the result count
) # Finish creating the structured query

visitor = MongoFilterVisitor() # Create the concrete visitor

mongo_request = structured_query.accept(visitor) # Dispatch the complete query to the visitor

print(mongo_request) # Display the translated MongoDB-style request

# `StructuredQuery: Expr`

Represents a query string with an optional filter and result limit.

## Fields

```python
query: str # Query string
filter: FilterDirective | None # Filtering expression
limit: int | None # Maximum number of results
```

## Constructor

```python
StructuredQuery(
    query: str, # Query string
    filter: FilterDirective | None, # Filtering expression
    limit: int | None = None, # Maximum number of results
    **kwargs: Any, # Additional values forwarded to BaseModel
) -> None
```

In [ ]:
from langchain_core.structured_query import Comparator, Comparison, StructuredQuery # Import structured-query classes


price_filter = Comparison( # Create a filter for the product price
    Comparator.LTE, # Select the less-than-or-equal comparator
    "price", # Specify the metadata attribute
    80000, # Set the maximum allowed price
) # Finish creating the comparison

structured_query = StructuredQuery( # Create the complete structured query
    "laptop suitable for software development", # Provide the semantic search query
    price_filter, # Attach the metadata filter
    limit=5, # Restrict the maximum number of results
) # Finish creating the structured query

print("Query:", structured_query.query) # Display the search query
print("Filter:", structured_query.filter) # Display the filtering expression
print("Limit:", structured_query.limit) # Display the maximum result count

In [ ]:
from langchain_core.structured_query import StructuredQuery # Import the structured-query model


simple_query = StructuredQuery( # Create a query without metadata filtering
    "beginner-friendly Python tutorials", # Provide the search query
    None, # Specify that no filter should be applied
    limit=3, # Restrict the maximum number of results
) # Finish creating the structured query

print(simple_query) # Display the complete structured-query object